<a href="https://colab.research.google.com/github/szwmya/Sentiment_Aware_Recommendation_Model/blob/main/final_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

from google.colab import drive
drive.mount('/content/drive')


data_path = '/content/drive/MyDrive/Amazon_dataset/Electronics_5.json'


In [ ]:
!pip install --upgrade transformers
!pip install huggingface_hub[hf_xet]

In [ ]:
import json
import pandas as pd

json_path = '/content/drive/MyDrive/Amazon_dataset/Electronics_5.json'


def parse_reviews_minimal(json_path):
    data = []

    def convert_score_to_label(score):
        if score >= 4:
            return "positive"
        elif score == 3:
            return "neutral"
        else:
            return "negative"

    with open(json_path, 'r') as f:
        for line in f:
            if line.strip():
                item = json.loads(line.strip())
                sentence = item.get("reviewText", "")
                score = item.get("overall", None)

                if sentence and score is not None:
                    sentiment = convert_score_to_label(score)
                    data.append({"sentence": sentence, "sentiment": sentiment})

    return pd.DataFrame(data)


Amazon_df = parse_reviews_minimal(json_path)
sample_df = Amazon_df.sample(n=50000, random_state=42).reset_index(drop=True)
print(sample_df.head())



In [ ]:
print(sample_df['sentiment'].value_counts())


In [ ]:
print("Original:", sample_df.shape)

Amazon_dd = sample_df.drop_duplicates()
dd = Amazon_dd.reset_index(drop=True)
print("Drop Duplicates:", dd.shape)

dd_dn = dd.dropna()
df = dd_dn.reset_index(drop=True)
print("Drop Nulls:", df.shape)

In [ ]:
import re

In [ ]:
def enhanced_clean_text(text):
    # Remove special patterns common in reviews
    text = re.sub(r'\b\d+ stars?\b', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\b(?:great|good|bad|excellent|poor)\b', '', text, flags=re.IGNORECASE)

    # Handle negations more carefully
    text = re.sub(r"\b(?:not|no|never)\s+(\w+)", r"not_\1", text)

    # Remove overly common uninformative phrases
    common_phrases = [
        "i bought this", "i purchased", "i recommend",
        "i would", "this product", "this item"
    ]
    for phrase in common_phrases:
        text = text.replace(phrase, '')

    return text

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
sample_df['label'] = label_encoder.fit_transform(sample_df['sentiment'])



In [ ]:

from transformers import RobertaTokenizer

tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

encodings = tokenizer.batch_encode_plus(
    sample_df['sentence'].tolist(),
    padding='max_length',
    truncation=True,
    max_length=100,
    return_attention_mask=True,
    return_tensors='pt'
)


In [2]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaTokenizer
from sklearn.preprocessing import LabelEncoder


label_encoder = LabelEncoder()
sample_df['label'] = label_encoder.fit_transform(sample_df['sentiment'])


tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

NameError: name 'sample_df' is not defined

In [ ]:
class ReviewDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=100):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = int(self.labels[idx])

        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        return {
            'text': text,
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label': torch.tensor(label, dtype=torch.long)
        }


In [ ]:
from imblearn.over_sampling import RandomOverSampler

# Before splitting the data
ros = RandomOverSampler(random_state=42)
X_resampled, y_resampled = ros.fit_resample(
    sample_df[['sentence']],
    sample_df['sentiment']
)

balanced_df = pd.DataFrame({
    'sentence': X_resampled['sentence'],
    'sentiment': y_resampled
})

# Now proceed with train/test split on balanced_df

In [ ]:

!pip install textblob
!pip install googletrans==4.0.0-rc1


In [ ]:
import random
from nltk.corpus import wordnet
import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')

def synonym_replacement(words, n):
    new_words = words.copy()
    random_word_list = list(set([word for word in words if wordnet.synsets(word)]))
    random.shuffle(random_word_list)
    num_replaced = 0
    for random_word in random_word_list:
        synonyms = wordnet.synsets(random_word)
        if synonyms:
            synonym = synonyms[0].lemmas()[0].name()
            new_words = [synonym if word == random_word else word for word in new_words]
            num_replaced += 1
        if num_replaced >= n:
            break
    return new_words

def augment_text(text, sentiment):
    words = text.split()
    n = max(1, int(0.1 * len(words)))  # replace 10% of words
    augmented_words = synonym_replacement(words, n)
    return ' '.join(augmented_words)


In [ ]:
from collections import Counter

# Count the original class distribution
print(Counter(sample_df['sentiment']))

# Augment negative and neutral samples
augmented_texts = []
augmented_labels = []

for _, row in df.iterrows():
    if row['sentiment'] in ['negative', 'neutral']:
        for _ in range(5):  # increase number as needed
            new_text = augment_text(row['sentence'], row['sentiment'])
            augmented_texts.append(new_text)
            augmented_labels.append(row['sentiment'])

# Create new DataFrame from augmented data
df_aug = pd.DataFrame({'sentence': augmented_texts, 'sentiment': augmented_labels})

# Combine with original data
df_balanced = pd.concat([df, df_aug], ignore_index=True)

# Check new distribution
print(Counter(df_balanced['sentiment']))


In [ ]:
from sklearn.model_selection import train_test_split
def collate_fn(batch):
    texts = [item['text'] if 'text' in item else item.get('raw_text') or "" for item in batch]
    labels = torch.stack([item['label'] for item in batch])

    return {
        'texts': texts,
        'label': labels
    }



train_val_texts, test_texts, train_val_labels, test_labels = train_test_split(
    sample_df['sentence'],
    sample_df['label'],
    test_size=0.2,
    stratify=sample_df['label'],
    random_state=42
)

train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_val_texts,
    train_val_labels,
    test_size=0.2,
    stratify=train_val_labels,
    random_state=42
)

train_dataset = ReviewDataset(train_texts.tolist(), train_labels.tolist(), tokenizer)
val_dataset = ReviewDataset(val_texts.tolist(), val_labels.tolist(), tokenizer)
test_dataset = ReviewDataset(test_texts.tolist(), test_labels.tolist(), tokenizer)


train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=32, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=32, collate_fn=collate_fn)



In [ ]:
import torch
import torch.nn as nn
from transformers import RobertaTokenizer, RobertaModel

class RobertaBiGRUModel(nn.Module):
    def __init__(self, hidden_size=256, reduced_dim=128, num_classes=3, dropout=0.4, fine_tune_layers=6):
        super(RobertaBiGRUModel, self).__init__()

        self.tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
        self.roberta = RobertaModel.from_pretrained("roberta-base")


        for param in self.roberta.parameters():
            param.requires_grad = True


        if fine_tune_layers > 0:
            for layer in roberta.encoder.layer[-3:]:
                for param in layer.parameters():
                    param.requires_grad = True


        self.bigru = nn.GRU(
            input_size=768,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )


        self.reduce_dim = nn.Sequential(
            nn.Linear(hidden_size * 2, reduced_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )


        self.classifier = nn.Sequential(
            nn.LayerNorm(reduced_dim),
            nn.Linear(reduced_dim, num_classes)
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, texts):
        encoded = self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors='pt'
        ).to(next(self.parameters()).device)


        with torch.set_grad_enabled(self.roberta.encoder.layer[-1].parameters().__next__().requires_grad):
            outputs = self.roberta(
                input_ids=encoded['input_ids'],
                attention_mask=encoded['attention_mask']
            )
        last_hidden_state = outputs.last_hidden_state


        gru_output, _ = self.bigru(last_hidden_state)
        pooled = torch.mean(gru_output, dim=1)


        reduced = self.reduce_dim(self.dropout(pooled))
        logits = self.classifier(reduced)
        return logits


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout=0.3):
        super(MultiHeadSelfAttention, self).__init__()
        assert embed_dim % num_heads == 0, "embed_dim must be divisible by num_heads"

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        self.values = nn.Linear(embed_dim, embed_dim, bias=False)
        self.keys = nn.Linear(embed_dim, embed_dim, bias=False)
        self.queries = nn.Linear(embed_dim, embed_dim, bias=False)
        self.fc_out = nn.Linear(embed_dim, embed_dim)

        self.attn_dropout = nn.Dropout(dropout)
        self.norm1 = nn.LayerNorm(embed_dim)

        # 🔹 Feedforward block after attention
        self.feedforward = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim * 4, embed_dim),
            nn.Dropout(dropout),
        )
        self.norm2 = nn.LayerNorm(embed_dim)

    def forward(self, x, mask=None):
        N, seq_len, _ = x.shape

        values = self.values(x)
        keys = self.keys(x)
        queries = self.queries(x)

        values = values.view(N, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        keys = keys.view(N, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        queries = queries.view(N, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        energy = torch.matmul(queries, keys.transpose(-2, -1)) / (self.head_dim ** 0.5)

        if mask is not None:
            energy = energy.masked_fill(mask == 0, float("-inf"))

        attention = F.softmax(energy, dim=-1)
        attention = self.attn_dropout(attention)

        out = torch.matmul(attention, values)
        out = out.transpose(1, 2).contiguous().view(N, seq_len, self.embed_dim)
        out = self.fc_out(out)

        # 🔸 Residual connection + normalization
        x = self.norm1(out + x)

        # 🔹 Feedforward + second residual
        ff_out = self.feedforward(x)
        x = self.norm2(ff_out + x)
        return x


In [ ]:
import pytest
import torch


def test_initialization():
    embed_dim = 256
    num_heads = 8
    model = MultiHeadSelfAttention(embed_dim, num_heads)

    # Test head dimension calculation
    assert model.head_dim == 32, "Incorrect head dimension calculation"

    # Test parameter shapes
    assert model.values.weight.shape == (embed_dim, embed_dim)
    assert model.feedforward[0].weight.shape == (embed_dim*4, embed_dim)

def test_forward_shape():
    batch_size = 4
    seq_len = 50
    embed_dim = 256
    num_heads = 8

    model = MultiHeadSelfAttention(embed_dim, num_heads)
    x = torch.randn(batch_size, seq_len, embed_dim)

    # Test basic forward pass
    output = model(x)
    assert output.shape == (batch_size, seq_len, embed_dim), \
        "Output shape mismatch"

    # Test multi-head reshaping
    queries = model.queries(x)
    assert queries.view(batch_size, seq_len, num_heads, -1).shape == \
        (batch_size, seq_len, num_heads, embed_dim//num_heads)

def test_mask_application():
    embed_dim = 128
    num_heads = 4
    model = MultiHeadSelfAttention(embed_dim, num_heads)

    # Create test input and mask
    x = torch.randn(2, 10, embed_dim)
    mask = torch.tril(torch.ones(10, 10)).unsqueeze(0)

    # Forward with mask
    output = model(x, mask=mask)

    # Verify masked positions get -inf attention
    energy = torch.matmul(model.queries(x), model.keys(x).transpose(-2, -1))
    masked_energy = energy.masked_fill(mask == 0, float("-inf"))
    assert torch.isinf(masked_energy).any(), "Mask not applied correctly"

def test_residual_connections():
    embed_dim = 512
    num_heads = 16
    model = MultiHeadSelfAttention(embed_dim, num_heads)

    # Create test input
    x = torch.randn(3, 25, embed_dim)
    original_x = x.clone()

    # Forward pass
    output = model(x)

    # Verify residual connection
    assert not torch.allclose(output, original_x), \
        "Residual connection failed - output equals input"
    assert torch.allclose(output, model.norm1(output)), \
        "LayerNorm not applied correctly"

def test_feedforward_block():
    embed_dim = 64
    num_heads = 2
    model = MultiHeadSelfAttention(embed_dim, num_heads)

    # Test feedforward dimension expansion
    ff_layer = model.feedforward
    test_input = torch.randn(5, embed_dim)
    expanded = ff_layer[0](test_input)
    assert expanded.shape == (5, embed_dim*4), \
        "Feedforward expansion incorrect"

    # Test full feedforward
    output = ff_layer(test_input)
    assert output.shape == (5, embed_dim), \
        "Feedforward final dimension incorrect"

def test_gradient_flow():
    embed_dim = 128
    num_heads = 4
    model = MultiHeadSelfAttention(embed_dim, num_heads)

    # Create test data
    x = torch.randn(2, 15, embed_dim, requires_grad=True)
    output = model(x)

    # Backprop test
    loss = output.sum()
    loss.backward()

    # Verify gradients exist
    assert x.grad is not None, "No gradients flowing to input"
    assert model.values.weight.grad is not None, \
        "No gradients in value weights"

In [ ]:
class DeepFMBlock(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, reduced_dim=128): # Pass reduced_dim as argument
        """
        A simple DeepFM block that models both linear and higher-order interactions.

        Args:
            input_dim (int): Dimension of input features.
            hidden_dim (int): Hidden dimension for the MLP part.
            reduced_dim (int): Dimensionality of the reduced representation.
        """
        super(DeepFMBlock, self).__init__()
        self.linear = nn.Linear(input_dim, 1)
        self.deepfm = nn.Sequential(
              nn.Linear(reduced_dim, 128), # Use reduced_dim here
              nn.BatchNorm1d(128),
              nn.GELU(),
              nn.Dropout(0.4), # Using a common dropout value, change if necessary
              nn.Linear(128, 64),
              nn.BatchNorm1d(64),
              nn.GELU(),
              nn.Dropout(0.4), # Using a common dropout value, change if necessary
              nn.Linear(64, 1)
           )

    def forward(self, x):
        linear_out = self.linear(x)
        deepfm_out = self.deepfm(x) # Changed mlp to deepfm for correctness
        return linear_out + deepfm_out # Changed mlp_out to deepfm_out
        print(linear_out.shape)
        print(deepfm_out.shape)

In [1]:
class FullRecommendationModel(nn.Module):
    def __init__(self, hidden_size=384, reduced_dim=256, num_heads=12, num_classes=3, dropout=0.3, fine_tune_layers=6):
        super().__init__()

        # Initialize tokenizer
        self.tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

        # Enhanced RoBERTa configuration
        self.roberta = RobertaModel.from_pretrained('roberta-base')

        # Freeze all layers first
        for param in self.roberta.parameters():
            param.requires_grad = False

        # Unfreeze last 6 layers (increased from 4)
        for layer in self.roberta.encoder.layer[-fine_tune_layers:]:
            for param in layer.parameters():
                param.requires_grad = True

        # Enhanced BiGRU with more capacity
        self.bigru = nn.GRU(
            input_size=768,
            hidden_size=hidden_size,
            num_layers=2,  # Increased capacity
            batch_first=True,
            bidirectional=True,
            dropout=0.3 if 2 > 1 else 0
        )

        # Improved dimension reduction
        self.dim_reduction = nn.Sequential(
            nn.Linear(hidden_size * 2, reduced_dim * 2),
            nn.GELU(),  # Better than ReLU
            nn.Dropout(dropout),
            nn.Linear(reduced_dim * 2, reduced_dim),
            nn.LayerNorm(reduced_dim)
        )

        # Enhanced Multi-head Attention
        self.mhsa = MultiHeadSelfAttention(
            embed_dim=reduced_dim,
            num_heads=num_heads,
            dropout=dropout
        )

        # Hierarchical Attention Mechanism
        self.word_attention = nn.Linear(reduced_dim, 1, bias=False)
        self.sent_attention = nn.Linear(reduced_dim, 1, bias=False)

        # Enhanced DeepFM Block
        self.deepfm = nn.Sequential(
            nn.Linear(reduced_dim, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1)
        )

        # Final Classifier with residual connection
        self.classifier = nn.Sequential(
            nn.Linear(1, reduced_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(reduced_dim, num_classes)
        )

        # Initialize weights properly
        self._init_weights()

    def _init_weights(self):
        for module in [self.bigru, self.dim_reduction, self.mhsa, self.deepfm, self.classifier]:
            if hasattr(module, 'weight') and module.weight is not None:
                if isinstance(module, nn.Linear):
                    nn.init.xavier_normal_(module.weight)
                    if module.bias is not None:
                        nn.init.constant_(module.bias, 0)
                elif isinstance(module, nn.GRU):
                    for name, param in module.named_parameters():
                        if 'weight' in name:
                            nn.init.orthogonal_(param)
                        elif 'bias' in name:
                            nn.init.constant_(param, 0)

    def forward(self, texts):
        # Tokenization
        encoded = self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors='pt'
        ).to(next(self.parameters()).device)

        # RoBERTa with gradient control
        with torch.set_grad_enabled(any(p.requires_grad for p in self.roberta.parameters())):
            outputs = self.roberta(
                input_ids=encoded['input_ids'],
                attention_mask=encoded['attention_mask']
            )
        roberta_out = outputs.last_hidden_state

        # Enhanced BiGRU processing
        gru_out, _ = self.bigru(roberta_out)

        # Dimension reduction
        reduced = self.dim_reduction(gru_out)

        # Hierarchical Attention
        word_weights = torch.softmax(self.word_attention(reduced), dim=1)
        word_context = torch.sum(reduced * word_weights, dim=1)

        sent_weights = torch.softmax(self.sent_attention(word_context.unsqueeze(1)), dim=1)
        context_vector = torch.sum(word_context * sent_weights, dim=1)

        # DeepFM
        deepfm_out = self.deepfm(context_vector)

        # Final classification
        logits = self.classifier(deepfm_out)
        return logits

NameError: name 'nn' is not defined

In [ ]:
from tqdm import tqdm



In [ ]:
torch.save(model.state_dict(), "sentiment_model.pth")
